#### Fully-connected Feed-forward Neural Network (FNN) for Analyzing Vessel Dilatation (heat maps) to obtain Aortic Aneurysm Contributors (Elastic Fiber Integrity and Mechanosensing). Architecture: The branch network takes vessel dilatation as input. The trunk network takes the co-ordinates in the z-$\theta$ plane. 

In [7]:
import jax.numpy as jnp
import numpy.random as npr
from jax import jit, grad, vmap
from jax.example_libraries.optimizers import adam
from jax import value_and_grad
from functools import partial
from jax import jacfwd, jacrev
import jax.nn as jnn
import math
from jax import random
import jax
import numpy as np
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from flax import linen as nn
import sklearn.metrics

import argparse
import os
import time
from termcolor import colored
from scipy.io import loadmat
import scipy.io as io
import pickle

import sys
sys.path.append("../..")

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9})
import seaborn as sns
sns.set_style("white")
sns.set_style("ticks")

import warnings
warnings.filterwarnings("ignore")

# Check where gpu is enable or not
from jax.lib import xla_bridge
print(xla_bridge.get_backend().platform)

cpu


In [8]:
cluster = False
save = True

In [ ]:
if cluster == True:
    parser = argparse.ArgumentParser()
    parser.add_argument('-seed', dest='seed', type=int, default=0, help='Seed number.')
    args = parser.parse_args()

    # Print all the arguments
    for arg in vars(args):
        print(f'{arg}: {getattr(args, arg)}')

    seed = args.seed

if cluster == False:
    seed = 0 # Seed number.

if save == True:
    resultdir = os.path.join(os.getcwd(), 'Results_pad', 'seed='+str(seed))
    if not os.path.exists(resultdir):
        os.makedirs(resultdir)

if save == True and cluster == True:
    orig_stdout = sys.stdout
    q = open(os.path.join(resultdir, 'output-'+'seed='+str(seed)+'.txt'), 'w')
    sys.stdout = q
    print ("------START------")

print('seed = '+str(seed))

seed = 0


In [10]:
np.random.seed(seed)
key = 1234 #random.PRNGKey(seed)

In [11]:
# Load the data
data = loadmat('../../data/mixed_data_heat_pad.mat') # Load the .mat file
locations = loadmat('../../data/new_delta=0000_pad.mat') # load the .mat file which contains the spatial locations for evaluation
# print(data)
print(data['F1_train'].shape) # Input dilatation
print(data['U1_train'].shape) # Output Elastic Fiber
print(data['U2_train'].shape) # Output Mechanosensing
print(locations['init_loc_cyl'].shape) # Spatial locations

# Convert NumPy arrays to PyTorch tensors
inputs1_train = jnp.array(data['F1_train']).reshape(-1,41*41)
outputs1_train = jnp.array(data['U1_train']).reshape(-1,41*41)
outputs2_train = jnp.array(data['U2_train']).reshape(-1,41*41)
inputs1_test = jnp.array(data['F1_test']).reshape(-1,41*41)
outputs1_test = jnp.array(data['U1_test']).reshape(-1,41*41)
outputs2_test = jnp.array(data['U2_test']).reshape(-1,41*41)

grid_theta = locations['init_loc_cyl'][:,0:1]
grid_z = locations['init_loc_cyl'][:,2:3]
grid = jnp.concatenate([grid_theta, grid_z], axis=1, dtype=None)
print("Shape of grid for Elastic Fiber and Mechanosensing:", grid.shape) # (nt*nx, 2)
print("grid:", grid) # theta, z

# Check the shapes of the subsets
print("Shape of inputs_train:", inputs1_train.shape)
print("Shape of inputs_test:", inputs1_test.shape)
print("Shape of outputs1_train (Elastic Fiber):", outputs1_train.shape)
print("Shape of outputs1_test (Elastic Fiber):", outputs1_test.shape)
print("Shape of outputs2_train (Mechanosensing):", outputs2_train.shape)
print("Shape of outputs2_test (Mechanosensing):", outputs2_test.shape)
print('#'*100)

(450, 41, 41)
(450, 41, 41)
(450, 41, 41)
(1681, 3)
Shape of grid for Elastic Fiber and Mechanosensing: (1681, 2)
grid: [[ 1.5707964   0.        ]
 [ 1.5707964   0.32821724]
 [ 1.5707964   0.6545085 ]
 ...
 [ 1.5707964   9.345491  ]
 [ 1.5707964   9.6717825 ]
 [ 1.5707964  10.        ]]
Shape of inputs_train: (450, 1681)
Shape of inputs_test: (50, 1681)
Shape of outputs1_train (Elastic Fiber): (450, 1681)
Shape of outputs1_test (Elastic Fiber): (50, 1681)
Shape of outputs2_train (Mechanosensing): (450, 1681)
Shape of outputs2_test (Mechanosensing): (50, 1681)
####################################################################################################


In [12]:
# Initialize the Glorot (Xavier) normal distribution for weight initialization
initializer = jax.nn.initializers.glorot_normal()

def init_glorot_params(layer_sizes, key = random.PRNGKey(seed)):
    """
    Initialize the parameters of the neural network using Glorot (Xavier) initialization.

    Args:
    layer_sizes (list): List of integers representing the size of each layer.
    key (PRNGKey): Random number generator key for reproducibility.

    Returns:
    list: List of tuples, each containing weights and biases for a layer.
    """
    return [(initializer(key, (m, n), jnp.float32), jnp.zeros(n))
            for m, n, in zip(layer_sizes[:-1], layer_sizes[1:])]

def BranchNet_dil(params, x):
    """
    Implement the branch network of the DeepONet.

    Args:
    params (list): List of weight and bias tuples for each layer.
    x (array): Input to the branch network.

    Returns:
    array: Output of the branch network.
    """
    def single_forward(params, x):
        for w, b in params:
            outputs = jnp.dot(x, w) + b
            x = jnn.silu(outputs)
        return outputs

    return vmap(partial(single_forward, params))(x)

def BranchNet_dis(params, x):
    """
    Implement the branch network of the DeepONet.

    Args:
    params (list): List of weight and bias tuples for each layer.
    x (array): Input to the branch network.

    Returns:
    array: Output of the branch network.
    """
    def single_forward(params, x):
        for w, b in params:
            outputs = jnp.dot(x, w) + b
            x = jnn.silu(outputs)
        return outputs

    return vmap(partial(single_forward, params))(x)

def TrunkNet(params, x, y):
    """
    Implement the trunk network of the DeepONet.

    Args:
    params (list): List of weight and bias tuples for each layer.
    x (float): First input to the trunk network.
    t (float): Second input to the trunk network.

    Returns:
    array: Output of the trunk network.
    """
    inputs = jnp.array([x, y])
    for w, b in params:
        outputs = jnp.dot(inputs, w) + b
        inputs = jnn.silu(outputs)
    return outputs

@jit
def DeepONet(params, branch_inputs_dil, trunk_inputs):
    """
    Implement the complete DeepONet architecture.

    Args:
    params (tuple): Tuple containing branch and trunk network parameters.
    branch_inputs (array): Inputs for the branch network.
    trunk_inputs (array): Inputs for the trunk network.

    Returns:
    array: Output of the DeepONet.
    """
    params_branch_dil, params_trunk = params
    branch_outputs_dil = lambda x: BranchNet_dil(params_branch_dil, x)
    b_out_dil = branch_outputs_dil(branch_inputs_dil)
    trunk_output_ef = lambda theta, z: TrunkNet(params_trunk, theta, z)
    t_out = vmap(trunk_output_ef)(trunk_inputs[:,0],trunk_inputs[:,1])
    results_ef = jnp.einsum('ik, lk -> il',b_out_dil[:,0:p], t_out[:,0:p])
    results_mech = jnp.einsum('ik, lk -> il',b_out_dil[:,p:2*p], t_out[:,p:2*p])
    return results_ef, results_mech

# network parameters.
p = 100 # Number of output neurons in both the branch and trunk net.
nx = 41*41
input_neurons_branch = nx # m
input_neurons_trunk = 2

layer_sizes_b_dil = [input_neurons_branch] + [100]*6 + [2*p]
layer_sizes_t = [input_neurons_trunk] + [100]*6 + [2*p]

params_branch_dil = init_glorot_params(layer_sizes=layer_sizes_b_dil)
params_trunk_ef = init_glorot_params(layer_sizes=layer_sizes_t)

params= (params_branch_dil, params_trunk_ef)

def objective(params, branch_inputs_dil, trunk_inputs, target_values_ef, target_values_mech):
    """
    Define the objective function (loss function) for training.

    Args:
    params (tuple): Tuple containing branch and trunk network parameters.
    branch_inputs (array): Inputs for the branch network.
    trunk_inputs (array): Inputs for the trunk network.
    target_values (array): True output values to compare against.

    Returns:
    float: Mean squared error loss.
    """
    predictions_ef, predictions_mech = DeepONet(params, branch_inputs_dil, trunk_inputs)
    loss_mse = jnp.mean((predictions_ef - target_values_ef)**2) + jnp.mean((predictions_mech - target_values_mech)**2)
    return loss_mse


# Adam optimizer
@jit
def resnet_update(params, branch_input_dil, trunk_inputs, target_values_ef, target_values_mech, opt_state):
    """
    Compute the gradient for a batch and update the parameters.

    Args:
    params (tuple): Current network parameters.
    branch_inputs (array): Inputs for the branch network.
    trunk_inputs (array): Inputs for the trunk network.
    target_values (array): True output values.
    opt_state: Current state of the optimizer.

    Returns:
    tuple: Updated parameters, updated optimizer state, and current loss value.
    """
    value, grads = value_and_grad(objective)(params, branch_input_dil, trunk_inputs, target_values_ef, target_values_mech)
    opt_state = opt_update(0, grads, opt_state)
    return get_params(opt_state), opt_state, value

# Initialize the Adam optimizer
opt_init, opt_update, get_params = adam(step_size=1e-3, b1=0.9, b2=0.999, eps=1e-08)
opt_state = opt_init(params)

bs = 450 #batch size
iteration_list, loss_list, test_loss_list = [], [], []
iteration = 0

n_epochs = 200000
num_samples = len(inputs1_train)

# test input preparation
branch_inputs1_test = inputs1_test
targets_ef = outputs1_test
targets_mech = outputs2_test

In [13]:
def save_model_params(params, resultdir, filename='model_params.pkl'):
    if not os.path.exists(resultdir):
        os.makedirs(resultdir)

    save_path = os.path.join(resultdir, filename)
    with open(save_path, 'wb') as f:
        pickle.dump(params, f)

def load_model_params(resultdir, filename='model_params.pkl'):
    load_path = os.path.join(resultdir, filename)
    with open(load_path, 'rb') as f:
        params = pickle.load(f)
    return params

# Saving
if save:
    save_model_params(params, resultdir)

# Loading (uncomment when needed)
# model_params = load_model_params(resultdir)

In [14]:
## Training of DeepONet
start = time.time() # start time of training
best_test_mse = float('inf')  # Initialize with infinity

# Save initial model at 0th iteration
save_model_params(params, resultdir, filename='model_params_best.pkl')
print("Saved initial model at iteration 0")

for iteration in range(n_epochs):
    indices = jax.random.permutation(jax.random.PRNGKey(0), num_samples)
    batch_index = indices[0:bs]
    inputs1_train_shuffled = inputs1_train[batch_index]
    outputs1_train_shuffled = outputs1_train[batch_index]
    outputs2_train_shuffled = outputs2_train[batch_index]
    target_values_ef = outputs1_train_shuffled
    target_values_mech = outputs2_train_shuffled
    branch_inputs_dil = inputs1_train_shuffled
    trunk_inputs = grid
    params, opt_state, value = resnet_update(params, branch_inputs_dil, trunk_inputs, target_values_ef, target_values_mech, opt_state)

    if iteration % 1000 == 0:
        params_branch, params_trunk = params
        predictions_ef, predictions_mech = DeepONet(params, branch_inputs_dil, trunk_inputs)
        test_mse = jnp.mean((predictions_ef - target_values_ef)**2) + jnp.mean((predictions_mech - target_values_mech)**2)

        # Compare current test error with the best so far
        if test_mse < best_test_mse:
            best_test_mse = test_mse
            # Save the model as it's the best so far
            save_model_params(params, resultdir, filename='model_params_best.pkl')
            print(f"New best model saved at iteration {iteration} with test MSE: {test_mse:.7f}")

        finish = time.time() - start
        print(f"Iteration: {iteration:3d}, Train loss: {objective(params, branch_inputs_dil, trunk_inputs, target_values_ef, target_values_mech):.7f}, Test loss: {test_mse:.7f}, Best test loss: {best_test_mse:.7f}, Time: {finish:.2f}")

    iteration_list.append(iteration)
    loss_list.append(objective(params, branch_inputs_dil, trunk_inputs, target_values_ef, target_values_mech))
    test_loss_list.append(test_mse)

if save:
    np.save(os.path.join(resultdir, 'iteration_list.npy'), np.asarray(iteration_list))
    np.save(os.path.join(resultdir, 'loss_list.npy'), np.asarray(loss_list))
    np.save(os.path.join(resultdir, 'test_loss_list.npy'), np.asarray(test_loss_list))

# Plotting code remains the same
plt.figure()
plt.plot(iteration_list, loss_list, 'g', label='Training loss')
plt.plot(iteration_list, test_loss_list, '-b', label='Test loss')
plt.yscale("log")
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

if save:
    plt.savefig(os.path.join(resultdir, 'loss_plot.pdf'))

# end timer
finish = time.time() - start
print("Time (sec) to complete:\n" + str(finish))

Saved initial model at iteration 0
New best model saved at iteration 0 with test MSE: 0.0829760
Iteration:   0, Train loss: 0.0829760, Test loss: 0.0829760, Best test loss: 0.0829760, Time: 0.90
New best model saved at iteration 1000 with test MSE: 0.0060692
Iteration: 1000, Train loss: 0.0060692, Test loss: 0.0060692, Best test loss: 0.0060692, Time: 20.21
New best model saved at iteration 2000 with test MSE: 0.0059569
Iteration: 2000, Train loss: 0.0059569, Test loss: 0.0059569, Best test loss: 0.0059569, Time: 39.94
New best model saved at iteration 3000 with test MSE: 0.0046721
Iteration: 3000, Train loss: 0.0046721, Test loss: 0.0046721, Best test loss: 0.0046721, Time: 59.27
New best model saved at iteration 4000 with test MSE: 0.0039151
Iteration: 4000, Train loss: 0.0039151, Test loss: 0.0039151, Best test loss: 0.0039151, Time: 78.39
New best model saved at iteration 5000 with test MSE: 0.0031675
Iteration: 5000, Train loss: 0.0031675, Test loss: 0.0031675, Best test loss: 0.0

KeyboardInterrupt: 

In [ ]:
# params_branch, params_trunk = params
# Load the best model parameters
best_params = load_model_params(resultdir, filename='model_params_best.pkl')
print("Loaded best model parameters")

# Predictions
mse_list_ef = []
mse_list_mech = []

branch_inputs_dil = inputs1_test
trunk_inputs = grid
prediction_ef, prediction_mech = DeepONet(best_params, branch_inputs_dil, trunk_inputs) # (bs, neval) = (1, nt*nz)

save_dict = {'dilatation': inputs1_test, 'pred_ef': prediction_ef, 'pred_mech': prediction_mech,\
             'target_ef': outputs1_test, 'target_mech': outputs2_test}

io.savemat(resultdir+'/pred.mat', save_dict)

for i in range(inputs1_test.shape[0]):

    branch_inputs_dil = inputs1_test[i].reshape(1, nx) # (bs, m) = (1, nt*nz)
    trunk_inputs = grid # (neval, 2) = (nt*nz, 2)

    prediction_i_ef, prediction_i_mech = DeepONet(best_params, branch_inputs_dil, trunk_inputs) # (bs, neval) = (1, nt*nz)
    target_i_ef = outputs1_test[i]
    target_i_mech = outputs2_test[i]

    mse_i_ef = np.mean((prediction_i_ef - target_i_ef)**2)
    mse_list_ef.append(mse_i_ef.item())
    mse_i_mech = np.mean((prediction_i_mech - target_i_mech)**2)
    mse_list_mech.append(mse_i_mech.item())

    if (i+1) % 1 == 0:
        print(colored('TEST SAMPLE '+str(i+1), 'red'))

        r2score_ef = metrics.r2_score(target_i_ef.flatten(), prediction_i_ef.flatten())
        relerror_ef = np.linalg.norm(target_i_ef- prediction_i_ef) / np.linalg.norm(target_i_ef)
        r2score_ef = float('%.4f'%r2score_ef)
        relerror_ef = float('%.4f'%relerror_ef)
        r2score_mech = metrics.r2_score(target_i_mech.flatten(), prediction_i_mech.flatten())
        relerror_mech = np.linalg.norm(target_i_mech- prediction_i_mech) / np.linalg.norm(target_i_mech)
        r2score_mech = float('%.4f'%r2score_mech)
        relerror_mech = float('%.4f'%relerror_mech)
        print('Rel. L2 Error (EF) = '+str(relerror_ef)+', R2 score (EF) = '+str(r2score_ef))
        print('Rel. L2 Error (Mech) = '+str(relerror_mech)+', R2 score (Mech) = '+str(r2score_mech))

        fig = plt.figure(figsize=(15, 8))

        # Adjust subplot parameters for better spacing
        plt.subplots_adjust(left=0.1, bottom=0.1, right=0.9, top=0.9, wspace=0.4, hspace=0.3)

        # Common input plot spanning both rows (1,1) and (2,1)
        ax = fig.add_subplot(2, 4, (1,5))
        inputs_print = inputs1_test[i].reshape(41, 41)
        plt.imshow(inputs_print)
        plt.colorbar()
        plt.title('$Dilatation$', fontsize=14)

        # First row - EF plots
        # True EF
        ax = fig.add_subplot(2, 4, 2)
        target_ef = target_i_ef.reshape(41, 41)
        plt.imshow(target_ef, cmap='jet')
        plt.colorbar()
        plt.title('$True \ EF \ field$', fontsize=14)

        # Predicted EF
        ax = fig.add_subplot(2, 4, 3)
        prediction_ef = prediction_i_ef.reshape(41, 41)
        plt.imshow(prediction_ef, cmap='jet')
        plt.colorbar()
        plt.title('$Predicted \ EF \ field$', fontsize=14)

        # Error EF
        ax = fig.add_subplot(2, 4, 4)
        error_ef = target_ef - prediction_ef
        plt.imshow(error_ef, cmap='jet')
        plt.colorbar()
        plt.title('$EF \ Absolute \ error$', fontsize=14)

        # Second row - Mech plots
        # True Mech
        ax = fig.add_subplot(2, 4, 6)
        target_mech = target_i_mech.reshape(41, 41)
        plt.imshow(target_mech, cmap='jet')
        plt.colorbar()
        plt.title('$True \ Mech \ field$', fontsize=14)

        # Predicted Mech
        ax = fig.add_subplot(2, 4, 7)
        prediction_mech = prediction_i_mech.reshape(41, 41)
        plt.imshow(prediction_mech, cmap='jet')
        plt.colorbar()
        plt.title('$Predicted \ Mech \ field$', fontsize=14)

        # Error Mech
        ax = fig.add_subplot(2, 4, 8)
        error_mech = target_mech - prediction_mech
        plt.imshow(error_mech, cmap='jet')
        plt.colorbar()
        plt.title('$Mech \ Absolute \ error$', fontsize=14)

        plt.tight_layout()

        if save == True:
            plt.savefig(os.path.join(resultdir,'Test_Sample_'+str(i+1)+'.pdf'))
            plt.show()
            plt.close()
        if save == False:
            plt.show()

        print(colored('#'*230, 'green'))

mse_ef = sum(mse_list_ef) / len(mse_list_ef)
print("Mean Squared Error Test (EF) :\n", mse_ef)
mse_mech = sum(mse_list_mech) / len(mse_list_mech)
print("Mean Squared Error Test (Mech) :\n", mse_mech)